In [125]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder


In [126]:
df = pd.read_csv('missing_imputeted_df.csv')
df.head()

,colony,property_type,price_outer,price_per_sqft,Total Floors,Property Age,bedrooms,bathrooms,balconies,built_up_area,study room,servant room,store room,pooja room,others,view floor plan,furnishing_type,luxury_score,Overlooking_Categories
0,bhopal,House,105.0,3750,1.0,Old Property,7,5,0,2800.0,0,0,0,0,0,0,1,71,Green View Homes
1,kolar road,Flat,85.0,6508,2.0,New Property,3,3,1,1567.2,0,0,0,1,0,1,1,35,Green View Homes
2,katara hills,Flat,25.0,2173,3.0,New Property,2,2,3,957.0,0,0,0,0,0,0,1,15,Green View Homes
3,aishbagh,House,65.0,5909,2.0,Old Property,4,5,1,2400.0,0,0,0,1,0,0,0,18,Road Facing Homes
4,chuna bhatti,House,650.0,16250,3.0,New Property,6,6,2,4000.0,1,1,1,1,0,0,2,10,Road Facing Homes


In [127]:
df.drop(columns=['price_per_sqft','study room','servant room','pooja room','others','view floor plan'],inplace=True)

In [128]:
df.head()

,colony,property_type,price_outer,Total Floors,Property Age,bedrooms,bathrooms,balconies,built_up_area,store room,furnishing_type,luxury_score,Overlooking_Categories
0,bhopal,House,105.0,1.0,Old Property,7,5,0,2800.0,0,1,71,Green View Homes
1,kolar road,Flat,85.0,2.0,New Property,3,3,1,1567.2,0,1,35,Green View Homes
2,katara hills,Flat,25.0,3.0,New Property,2,2,3,957.0,0,1,15,Green View Homes
3,aishbagh,House,65.0,2.0,Old Property,4,5,1,2400.0,0,0,18,Road Facing Homes
4,chuna bhatti,House,650.0,3.0,New Property,6,6,2,4000.0,1,2,10,Road Facing Homes


In [129]:
# ohe -> colony,property_type,overlooking category
# ordinal -> property age
def luxury_category(value):
    if value>0 and value <=25:
        return 0
    elif value>25 and value <=75: 
        return 1
    else:
        return 2
df['luxury_score'] = df['luxury_score'].apply(luxury_category)

In [130]:
df['Property Age'] =  df['Property Age'].replace({'Under Construction':0,'New Property':1,'Moderate Property':2,'Old Property':3})

In [131]:
df.head()

,colony,property_type,price_outer,Total Floors,Property Age,bedrooms,bathrooms,balconies,built_up_area,store room,furnishing_type,luxury_score,Overlooking_Categories
0,bhopal,House,105.0,1.0,3,7,5,0,2800.0,0,1,1,Green View Homes
1,kolar road,Flat,85.0,2.0,1,3,3,1,1567.2,0,1,1,Green View Homes
2,katara hills,Flat,25.0,3.0,1,2,2,3,957.0,0,1,0,Green View Homes
3,aishbagh,House,65.0,2.0,3,4,5,1,2400.0,0,0,0,Road Facing Homes
4,chuna bhatti,House,650.0,3.0,1,6,6,2,4000.0,1,2,0,Road Facing Homes


In [132]:
df = pd.get_dummies(df,columns=['property_type','colony','Overlooking_Categories'],drop_first=True,dtype='int')
df.head()

,price_outer,Total Floors,Property Age,bedrooms,bathrooms,balconies,built_up_area,store room,furnishing_type,luxury_score,...,colony_patel nagar,colony_peer gate area,colony_piplani,colony_salaiya,colony_shahjahanabad,colony_shahpura,colony_shirdipuram,Overlooking_Categories_Green View Homes,Overlooking_Categories_Premium Scenic Homes,Overlooking_Categories_Road Facing Homes
0,105.0,1.0,3,7,5,0,2800.0,0,1,1,...,0,0,0,0,0,0,0,1,0,0
1,85.0,2.0,1,3,3,1,1567.2,0,1,1,...,0,0,0,0,0,0,0,1,0,0
2,25.0,3.0,1,2,2,3,957.0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
3,65.0,2.0,3,4,5,1,2400.0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,650.0,3.0,1,6,6,2,4000.0,1,2,0,...,0,0,0,0,0,0,0,0,0,1


In [133]:
inputs = df.drop(columns='price_outer')
output = df['price_outer']

In [134]:
output_log = np.log1p(output)

In [135]:
scaler = StandardScaler()

inputs_scaled = scaler.fit_transform(inputs)
inputs_scaled

array([[-0.86411934,  1.43795184,  2.95034513, ...,  1.47966529,
        -0.18716234, -0.91897931],
       [-0.31347402, -0.99252046, -0.06056427, ...,  1.47966529,
        -0.18716234, -0.91897931],
       [ 0.23717131, -0.99252046, -0.81329162, ...,  1.47966529,
        -0.18716234, -0.91897931],
       ...,
       [-0.31347402,  0.22271569,  0.69216308, ..., -0.67582852,
        -0.18716234, -0.91897931],
       [-0.31347402, -0.99252046,  1.44489043, ..., -0.67582852,
        -0.18716234, -0.91897931],
       [ 0.78781664,  0.22271569, -0.06056427, ..., -0.67582852,
        -0.18716234, -0.91897931]])

In [136]:
inputs_df = pd.DataFrame(inputs_scaled,columns=inputs.columns)

In [137]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), inputs_scaled, output_log, cv=kfold, scoring='r2')

In [138]:
scores.mean(),scores.std()

(0.6887546207952665, 0.06522282203200251)

In [139]:
lr = LinearRegression()
lr.fit(inputs_scaled,output_log)
coef_df = pd.DataFrame(lr.coef_.reshape(1,49),columns=inputs.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [142]:
def get_insight(feature_name):

    idx = list(inputs_df.columns).index(feature_name)
    
    coef = lr.coef_[idx]
    std = scaler.scale_[idx]

    real_coef = coef / std
    factor = np.exp(real_coef)

    base_price = df['price_outer'].median()

    rupee_change = base_price * (factor - 1)
    rupees = rupee_change * 100000

    # 🔥 FIX: use rupees consistently
    if rupees > 0:
        value = round(rupees)
        return f"1 unit increase in {feature_name} → price increases by approx ₹{value}"
    else:
        value = round(abs(rupees))
        return f"1 unit decrease in {feature_name} → price decreases by approx ₹{value}"

In [154]:
get_insight('Overlooking_Categories_Road Facing Homes')

'1 unit decrease in Overlooking_Categories_Road Facing Homes → price decreases by approx ₹698598'

In [42]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(inputs_df)

# 3. Fit the model
model = sm.OLS(output_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            price_outer   R-squared:                       0.723
Model:                            OLS   Adj. R-squared:                  0.714
Method:                 Least Squares   F-statistic:                     80.62
Date:                Sat, 11 Apr 2026   Prob (F-statistic):               0.00
Time:                        16:49:07   Log-Likelihood:                -994.27
No. Observations:                1566   AIC:                             2089.
Df Residuals:                    1516   BIC:                             2356.
Df Model:                          49                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [62]:
# y_log_std = output_log.std()
# # y_log_std

In [59]:
# coef = coef_df[coef_df['feature']=='built_up_area']['coef']
# coef

In [60]:
# std = inputs_df['built_up_area'].std()
# std

In [61]:
# np.expm1(coef*(y_log_std/std))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# ================================
# LOAD DATA
# ================================
df = pd.read_csv("your_data.csv")

# ================================
# ENCODING
# ================================
df = pd.get_dummies(
    df,
    columns=['property_type', 'colony', 'Overlooking_Categories'],
    drop_first=True
)

# ================================
# SPLIT
# ================================
X = df.drop(columns=['price_outer'])
y = np.log(df['price_outer'])   # ✅ LOG MODEL

# ================================
# SCALE
# ================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ================================
# TRAIN
# ================================
model = LinearRegression()
model.fit(X_scaled, y)

# ================================
# INSIGHT FUNCTION
# ================================
def get_insight(feature_name):

    idx = list(X.columns).index(feature_name)
    
    coef = model.coef_[idx]
    std = scaler.scale_[idx]

    # remove scaling
    real_coef = coef / std

    # multiplicative effect
    factor = np.exp(real_coef)

    # base price (median better)
    base_price = df['price_outer'].median()

    # rupee change
    rupee_change = base_price * (factor - 1)

    if rupee_change > 0:
        return f"1 unit increase in {feature_name} → price increases by approx ₹{int(rupee_change)}"
    else:
        return f"1 unit increase in {feature_name} → price decreases by approx ₹{int(abs(rupee_change))}"